# 02 — Engagement Rate Predictor (MLP)

Train and evaluate the feed-forward neural network engagement rate predictor. This notebook walks through architecture design, training, and evaluation.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, random_split
from services.ml.app.models.engagement import EngagementMLP
from services.ml.training.generate_synthetic_data import generate_engagement_data

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
X, y = generate_engagement_data(n=20000)
X_t  = torch.tensor(X, dtype=torch.float32)
y_t  = torch.tensor(y, dtype=torch.float32)

ds               = TensorDataset(X_t, y_t)
n_val            = int(0.15 * len(ds))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val])
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
model   = EngagementMLP().to(device)
optim   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched   = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=80)
loss_fn = nn.MSELoss()

train_ldr = DataLoader(train_ds, batch_size=256, shuffle=True)
val_ldr   = DataLoader(val_ds,   batch_size=512)

train_losses, val_losses = [], []
EPOCHS = 80

for epoch in range(1, EPOCHS + 1):
    model.train()
    tl = 0
    for xb, yb in train_ldr:
        xb, yb = xb.to(device), yb.to(device)
        optim.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optim.step()
        tl += loss.item() * len(xb)
    train_losses.append(tl / len(train_ds))
    sched.step()

    model.eval()
    vl = 0
    with torch.no_grad():
        for xb, yb in val_ldr:
            xb, yb = xb.to(device), yb.to(device)
            vl += loss_fn(model(xb), yb).item() * len(xb)
    val_losses.append(vl / len(val_ds))

    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | train_rmse={train_losses[-1]**0.5:.5f} | val_rmse={val_losses[-1]**0.5:.5f}')

print('Training complete.')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot([l**0.5 for l in train_losses], label='Train RMSE', color='#405DE6')
plt.plot([l**0.5 for l in val_losses],   label='Val RMSE',   color='#E1306C')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('Engagement Predictor — Training Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final val RMSE: {val_losses[-1]**0.5:.5f}')

In [ ]:
model.eval()
X_val = X_t[train_ds.indices[0]:train_ds.indices[0]+1000] if hasattr(train_ds, 'indices') else X_t[:1000]
y_val = y_t[:1000]
with torch.no_grad():
    preds = model(X_val.to(device)).cpu().numpy()

plt.figure(figsize=(8, 6))
plt.scatter(y_val.numpy(), preds, alpha=0.3, s=10, color='#833AB4')
plt.plot([0, 0.3], [0, 0.3], 'r--', linewidth=2, label='Perfect')
plt.xlabel('Actual Engagement Rate')
plt.ylabel('Predicted Engagement Rate')
plt.title('Predicted vs Actual Engagement Rate')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()